# Training and Evaluation
Trains a U-Net (ResNet34 encoder) on the patch archive from the data preparation notebook,
evaluates on a held-out test set, and saves the best model to Drive.

**Runtime:** T4 GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import tarfile, os

DRIVE_SAVE_DIR = '/content/drive/MyDrive/Gala_datasets'
archive_path = os.path.join(DRIVE_SAVE_DIR, 'split.tar.gz')
extract_path = '/content/data'

with tarfile.open(archive_path, 'r:gz') as t:
    t.extractall(extract_path)

for split in ['train', 'val', 'test']:
    imgs = os.listdir(os.path.join(extract_path, 'split', split, 'images'))
    masks = os.listdir(os.path.join(extract_path, 'split', split, 'masks'))
    print(f'{split}: {len(imgs)} images, {len(masks)} masks')

In [ ]:
!pip install segmentation-models-pytorch albumentations rasterio -q

In [ ]:
import os, glob, shutil
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
import rasterio
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__} | Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)')

In [ ]:
DATA_ROOT = '/content/data/split'
DRIVE_SAVE_DIR = '/content/drive/MyDrive/Gala_datasets'

ENCODER = 'resnet34'
IN_CHANNELS = 4
BATCH_SIZE = 16
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4

print('Configuration loaded')

## Dataset and data loading

In [ ]:
class GalamseyDataset(Dataset):
    def __init__(self, image_dir, mask_dir, augment=False):
        self.image_paths = sorted(glob.glob(os.path.join(image_dir, '*.tif')))
        self.mask_dir = mask_dir
        self.augment = augment
        self.transform = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.RandomBrightnessContrast(p=0.3),
        ]) if augment else None

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        with rasterio.open(self.image_paths[idx]) as src:
            image = src.read().astype(np.float32)

        fname = os.path.basename(self.image_paths[idx])
        mask_path = os.path.join(self.mask_dir, fname.replace('.tif', '_mask.tif'))
        with rasterio.open(mask_path) as src:
            mask = src.read(1).astype(np.int64)

        image = np.nan_to_num(image, nan=0.0, posinf=0.0, neginf=0.0)
        image = np.clip(image / 10000.0, 0, 1)  # Planet NICFI reflectance to 0-1

        if self.transform:
            aug = self.transform(image=image.transpose(1, 2, 0), mask=mask)
            image = aug['image'].transpose(2, 0, 1)
            mask = aug['mask']

        return torch.tensor(image), torch.tensor(mask)

In [ ]:
train_dataset = GalamseyDataset(os.path.join(DATA_ROOT, 'train', 'images'), os.path.join(DATA_ROOT, 'train', 'masks'), augment=True)
val_dataset = GalamseyDataset(os.path.join(DATA_ROOT, 'val', 'images'), os.path.join(DATA_ROOT, 'val', 'masks'), augment=False)
test_dataset = GalamseyDataset(os.path.join(DATA_ROOT, 'test', 'images'), os.path.join(DATA_ROOT, 'test', 'masks'), augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

images, masks = next(iter(train_loader))
print(f'Batch: {images.shape} | Mask: {masks.shape}')
print(f'Range: {images.min():.3f} to {images.max():.3f}')
print(f'Mask values: {torch.unique(masks).tolist()}')

## Model, loss, and optimizer

In [ ]:
model = smp.Unet(
    encoder_name=ENCODER,
    encoder_weights='imagenet',
    in_channels=IN_CHANNELS,
    classes=1,
).to(device)

In [ ]:
class CombinedLoss(nn.Module):
    """BCE + Dice loss. Pixels with mask value 255 (unlabelled) are excluded."""
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, pred, target):
        pred = pred.squeeze(1)
        target = target.long()
        valid = (target != 255).float()
        if valid.sum() == 0:
            return torch.tensor(0.0, device=pred.device, requires_grad=True)
        target_float = target.float().clamp(0, 1)
        bce_loss = (self.bce(pred, target_float) * valid).sum() / valid.sum()
        pred_sig = torch.sigmoid(pred)
        pred_valid = pred_sig * valid
        target_valid = target_float * valid
        intersection = (pred_valid * target_valid).sum()
        union = pred_valid.sum() + target_valid.sum()
        dice_loss = 1 - (2 * intersection + 1) / (union + 1)
        return bce_loss + dice_loss

criterion = CombinedLoss()

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

## Training loop

In [ ]:
best_val_loss = float('inf')
train_losses, val_losses = [], []

print(f'{"Epoch":>6} {"Train Loss":>12} {"Val Loss":>12} {"LR":>10} {"Status":>10}')
print('-' * 55)

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    for imgs, msks in train_loader:
        imgs, msks = imgs.to(device), msks.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), msks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    model.eval()
    running_val = 0.0
    with torch.no_grad():
        for imgs, msks in val_loader:
            imgs, msks = imgs.to(device), msks.to(device)
            running_val += criterion(model(imgs), msks).item()
    val_loss = running_val / len(val_loader)
    val_losses.append(val_loss)

    scheduler.step(val_loss)
    lr = optimizer.param_groups[0]['lr']

    status = ''
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), '/content/best_model.pth')
        status = 'saved'

    print(f'{epoch+1:>6} {train_loss:>12.4f} {val_loss:>12.4f} {lr:>10.6f} {status:>10}')

print(f'\nBest val loss: {best_val_loss:.4f}')

## Evaluation and visualization

In [ ]:
# Load best checkpoint and evaluate on the held-out test set
model.load_state_dict(torch.load('/content/best_model.pth', weights_only=True))
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for imgs, msks in test_loader:
        preds = (torch.sigmoid(model(imgs.to(device)).squeeze(1)) > 0.5).cpu().numpy()
        msks = msks.numpy()
        for p, m in zip(preds, msks):
            valid = m != 255  # only evaluate on labelled pixels
            all_preds.append(p[valid])
            all_targets.append(m[valid])

all_preds = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

tp = ((all_preds == 1) & (all_targets == 1)).sum()
fp = ((all_preds == 1) & (all_targets == 0)).sum()
fn = ((all_preds == 0) & (all_targets == 1)).sum()
tn = ((all_preds == 0) & (all_targets == 0)).sum()

precision = tp / (tp + fp + 1e-8)
recall = tp / (tp + fn + 1e-8)
f1 = 2 * precision * recall / (precision + recall + 1e-8)
iou = tp / (tp + fp + fn + 1e-8)

print('Test Set Evaluation')
print('=' * 45)
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1 Score:  {f1:.4f}')
print(f'IoU:       {iou:.4f}')
print(f'\nTP: {tp:,}  FP: {fp:,}  FN: {fn:,}  TN: {tn:,}')

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(14, 18))
axes[0][0].set_title('Satellite (RGB)', fontsize=12)
axes[0][1].set_title('True mask', fontsize=12)
axes[0][2].set_title('Predicted mask', fontsize=12)

for row, idx in enumerate(np.random.choice(len(test_dataset), 4, replace=False)):
    image, mask = test_dataset[idx]
    with torch.no_grad():
        pred = torch.sigmoid(model(image.unsqueeze(0).to(device))).squeeze().cpu().numpy()
    pred_bin = (pred > 0.5).astype(np.uint8)
    mask_np = mask.numpy()

    rgb = np.clip(image[[2,1,0]].numpy() * 3, 0, 1)
    axes[row][0].imshow(rgb.transpose(1,2,0)); axes[row][0].axis('off')

    mask_disp = np.zeros((*mask_np.shape, 3))
    mask_disp[mask_np == 1] = [0,1,0]; mask_disp[mask_np == 255] = [0.3,0.3,0.3]
    axes[row][1].imshow(mask_disp); axes[row][1].axis('off')

    pred_disp = np.zeros((*pred_bin.shape, 3))
    pred_disp[pred_bin == 1] = [0,1,0]
    axes[row][2].imshow(pred_disp); axes[row][2].axis('off')

plt.tight_layout()
plt.show()

## Save model

In [ ]:
shutil.copy2('/content/best_model.pth', os.path.join(DRIVE_SAVE_DIR, 'best_model.pth'))
print(f'Model saved to: {DRIVE_SAVE_DIR}/best_model.pth')

---
**Resume from saved model** — run the setup and data loading cells above, then uncomment and run this cell. Skip training.

In [ ]:
# model = smp.Unet(encoder_name=ENCODER, encoder_weights=None, in_channels=IN_CHANNELS, classes=1)
# model.load_state_dict(torch.load(os.path.join(DRIVE_SAVE_DIR, 'best_model.pth'), weights_only=True))
# model = model.to(device)
# model.eval()
# print('Model loaded from Drive')